In [1]:
import functools, pickle, gin, pathlib, time

import jax, jax.numpy as jnp
import haiku as hk
import matplotlib.pyplot as plt
import seaborn; seaborn.set()

import jax_cfd.base as cfd
from jax_cfd.base import grids, boundaries, pressure, finite_differences as fd
from jax_cfd.base.boundaries import (
    channel_flow_with_simple_immersed_body_boundary_conditions as cf_ib_bc
)
from jax_cfd.ml import model_builder, physics_specifications as phys
jax.devices()          # make sure you see a GPU

[CudaDevice(id=0)]

In [2]:
# =========================================
# 1.  Domain, grid and physical BCs
# =========================================
# Channel 8×2 with a circular cylinder in mid-channel

NX, NY        = 512, 128
DOMAIN        = ((0., 8.), (0., 2.))
DENSITY       = 1.0
NU            = 1e-3                       # kinematic viscosity
DPDX          = 2e-3                       # driving pressure gradient
grid          = grids.Grid((NX, NY), domain=DOMAIN)

# Cylinder parameters – centre and radius (in physical coords!)
CYL_CX, CYL_CY, CYL_R = 2.0, 1.0, 0.25

# Dirichlet walls (top & bottom) + inflow/outflow in x + cylinder mask
vel_bc = (
    cf_ib_bc(grid, shape='circle', shape_size=CYL_R, bc_value=0.0),
    cf_ib_bc(grid, shape='circle', shape_size=CYL_R, bc_value=0.0),
)

In [3]:
# --- helpers to pad to full interior size and unpad back --------------
def pad_to_grid(arr, offset):
    """Pad a staggered field so shape matches interior + 1 if offset == 0.5."""
    pads = []
    for ax, off in enumerate(offset):
        pad_amt = 1 if jnp.isclose(off % 1, 0.5) else 0
        pads.append((0, pad_amt))
    return jnp.pad(arr, pads)

def pad_to_grid(field):
    data = field.data
    for axis, off in enumerate(field.offset):
        if off % 1 == 0.5:  # only pad staggered axes (offset with fractional 0.5)
            pad_cfg = [(0, 0)] * data.ndim
            pad_cfg[axis] = (0, 1)  # add one point on the upper end of this axis
            data = jnp.pad(data, pad_cfg, mode='constant', constant_values=0.0)
    return data


def unpad(arr, original_shape):
    """Trim back to the original staggered size."""
    slices = tuple(slice(0, s) for s in original_shape)
    return arr[slices]


In [4]:
# =========================================
# 2.  Initial condition  (fluid initially at rest)
# =========================================
zero_fn   = lambda x, y: jnp.zeros_like(x + y)
v0        = cfd.initial_conditions.initial_velocity_field(
    velocity_fns=(zero_fn, zero_fn),
    grid=grid,
    velocity_bc=vel_bc,
    pressure_solve=cfd.pressure.solve_fast_diag_channel_flow,  # periodic solver OK for IC build
    iterations=5,
)

In [5]:
# =========================================
# 3.  Cylinder MASK on staggered grids
#      (hard-zero after ML prediction)
# =========================================
# cell-centres coordinates
xc, yc = grid.axes()
XC, YC = jnp.meshgrid(xc, yc, indexing='ij')
mask_center = ( (XC - CYL_CX)**2 + (YC - CYL_CY)**2 <= CYL_R**2 ).astype(jnp.float32)

# staggered shapes differ by 1 along one axis → quick pad/slice
mask_u = jnp.pad(mask_center, ((0, 0), (0, 1)))[:, :-1]   # matches u-x shape
mask_v = jnp.pad(mask_center, ((0, 1), (0, 0)))[:-1, :]   # matches v-y shape

In [ ]:
# =========================================
# 4.  Load learned-interpolation checkpoint
# =========================================

import sys
# Dummy placeholder for checkpointstate
sys.modules['__main__'].CheckpointState = type('CheckpointState', (), {})

CKPT_PATH = pathlib.Path("~/jax_cfd_models/LI/LI_ckpt.pkl").expanduser()
with CKPT_PATH.open("rb") as fp:
    ckpt = pickle.load(fp)

params        = ckpt.eval_params
dt            = ckpt.model_time_step        # → matches training value (e.g. 1e-3)
gin.clear_config()
gin.parse_config(ckpt.model_config_str)
physics_specs = phys.get_physics_specs()

# build model class & wrap into haiku transform
model_cls = model_builder.get_model_cls(grid, dt, physics_specs)

@hk.without_apply_rng
@hk.transform
def one_step(u, v):          # u,v arrays (no batch dim)
    solver = model_cls()
    s  = solver.encode((u, v))
    s  = solver.advance(s)    # learned interpolation + native diffusion
    out_u, out_v = solver.decode(s)
    return out_u, out_v       # still plain jnp arrays




# initialise params-shape dummy once (not used because we loaded real params)
dummy_u = pad_to_grid(v0[0])
dummy_v = pad_to_grid(v0[1])


print("dummy_u.shape:", dummy_u.shape, " offset:", v0[0].offset, " grid:", v0[0].grid.shape)
print("dummy_v.shape:", dummy_v.shape, " offset:", v0[1].offset, " grid:", v0[1].grid.shape)
_        = one_step.init(jax.random.PRNGKey(0), dummy_u, dummy_v)

apply_fn = functools.partial(one_step.apply, params)      # pure function (u,v)→(u,v)

AttributeError: 'jaxlib.xla_extension.ArrayImpl' object has no attribute 'data'

In [ ]:
# =========================================
# 5.  Step function:  ML  →  IBM  →  enforce BC  →  projection
# =========================================
def impose_walls(vel_tuple):
    return tuple(bc.impose_bc(v) for v, bc in zip(vel_tuple, vel_bc))

def hard_mask(vel_tuple):
    u, v = vel_tuple
    return (
        u * (1 - mask_u),            # zero inside cylinder
        v * (1 - mask_v),
    )

def ml_ibm_step(v_tuple, _):
    # --- 1) pad staggered arrays so they look like full interior ------------
    u_raw, v_raw  = v_tuple                     # GridArray objects
    u_in  = pad_to_grid(u_raw.data)
    v_in  = pad_to_grid(v_raw.data)

    # --- 2) ML prediction (learned-interp Δt) -------------------------------
    u_pred, v_pred = apply_fn(u_in, v_in)

    # --- 3) unpad back to staggered shapes ----------------------------------
    u_pred = unpad(u_pred, u_raw.shape)
    v_pred = unpad(v_pred, v_raw.shape)
    pred   = (
        grids.GridArray(u_pred, u_raw.offset, u_raw.grid),
        grids.GridArray(v_pred, v_raw.offset, v_raw.grid),
    )

    # --- 4) physics corrections ---------------------------------------------
    pred = hard_mask(pred)                 # cylinder no-slip
    pred = impose_walls(pred)              # channel walls / inflow/outflow
    pred = pressure.projection(            # divergence-free  (CG solver)
        pred, solve=pressure.solve_cg
    )
    return pred, None

ml_ibm_step_jit = jax.jit(ml_ibm_step)


In [ ]:
# =========================================
# 6.  Rollout
#     inner_steps  = dt steps between outputs
#     outer_steps  = number of outputs
# =========================================
N_INNER, N_OUTER = 400, 25        # → ≈ small demo; raise for long sim

stepper   = functools.partial(
    cfd.funcutils.repeated(ml_ibm_step_jit, N_INNER), None
)
trajectory_fn = jax.jit(
    cfd.funcutils.trajectory(stepper, N_OUTER, start_with_input=True)
)

print("JIT-compiling the rollout …")
t0 = time.time()
_, traj = trajectory_fn(v0)       # tuple of (u,v) with length=N_OUTER+1
jax.device_get(traj)              # materialise to host
print(f"Done, wall-time {time.time() - t0:.1f}s.")

In [ ]:
# =========================================
# 7.  Quick sanity visualisation
# =========================================
u_traj = jnp.stack([arr.data for arr in traj[0]])   # (N+1, nx, ny-1)
v_traj = jnp.stack([arr.data for arr in traj[1]])   # (N+1, nx-1, ny)

# centre the staggered velocity to cell centres for plotting
u_c = 0.5 * (u_traj[..., :, :-1] + u_traj[..., :, 1:])
v_c = 0.5 * (v_traj[..., :-1, :] + v_traj[..., 1:, :])
speed = jnp.sqrt(u_c**2 + v_c**2)

plt.figure(figsize=(10, 2.5))
plt.imshow(speed[-1].T, origin='lower', cmap='turbo',
           extent=[*DOMAIN[0], *DOMAIN[1]], aspect='auto')
circle = plt.Circle((CYL_CX, CYL_CY), CYL_R, color='k')
plt.gca().add_patch(circle)
plt.title('Speed field at final output step')
plt.colorbar(); plt.tight_layout()